In [ ]:
import os, math, random, json, hashlib
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (accuracy_score, f1_score, classification_report,
                             ConfusionMatrixDisplay, roc_curve, auc)
from sklearn.preprocessing import label_binarize
from transformers import XLMRobertaTokenizer, XLMRobertaModel
import matplotlib.pyplot as plt

DATA = Path('Wata.csv')
MODEL = os.getenv('SORANI_CHECKPOINT', './sorani-xlmr-pretrained-checkpoint-15-epoch')
OUT = Path('wata_training_results')
SEEDS = [42, 123, 777, 2024, 3407]
LABELS = ['Normal','Romantic','Advice','Threat','Hate Speech']
EPOCHS, MAX_LEN, TRAIN_BS, EVAL_BS = 10, 48, 100, 32
SPLIT_SEED, ENCODER_LR, HEAD_LR = 42, 1e-5, 5.5e-5
WEIGHT_DECAY, WARMUP, SMOOTHING = 0.012, 0.04, 0.05
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
assert DATA.is_file(), 'Wata.csv is required'
if MODEL.startswith('./'):
    assert (Path(MODEL)/'config.json').is_file(), 'Place the actual adapted checkpoint next to this notebook'
    assert any((Path(MODEL)/x).is_file() for x in ('model.safetensors','pytorch_model.bin')), 'Adapted model weights are missing'
OUT.mkdir(exist_ok=True)


In [ ]:
df = pd.read_csv(DATA)
assert len(df)==15052 and {'text','label'}.issubset(df.columns)
assert not df[['text','label']].isna().any().any()
df = df[['text','label']].copy()
df['label'] = df['label'].astype(int)
assert set(df.label.unique())==set(range(5))
df['_source_row'] = np.arange(len(df))
trainval = df.sample(frac=0.8, random_state=SPLIT_SEED)
train, val = train_test_split(trainval, test_size=0.10,
                              stratify=trainval['label'], random_state=SPLIT_SEED)
test = df.drop(trainval.index)
train, val, test = (x.reset_index(drop=True) for x in (train,val,test))
assert (len(train), len(val), len(test)) == (10837,1205,3010)

for name, frame in [('train',train),('validation',val),('test',test)]:
    saved = Path('splits')/f'{name}_split.csv'
    if saved.exists():
        original = pd.read_csv(saved)
        assert original['_source_row'].tolist()==frame['_source_row'].tolist(), f'{name} row-order mismatch'
        assert original[['text','label']].equals(frame[['text','label']]), f'{name} content mismatch'
    else:
        (OUT/'splits').mkdir(exist_ok=True)
        frame.to_csv(OUT/'splits'/f'{name}_split.csv',index=False,encoding='utf-8-sig')

weights = compute_class_weight('balanced',classes=np.arange(5),y=train.label.to_numpy())
print('Partition sizes:',len(train),len(val),len(test),'Class weights:',weights.round(4))
tokenizer = XLMRobertaTokenizer.from_pretrained(MODEL)


In [ ]:
class WataDataset(Dataset):
    def __init__(self, frame):
        self.frame = frame.reset_index(drop=True)
    def __len__(self):
        return len(self.frame)
    def __getitem__(self, i):
        row = self.frame.iloc[i]
        t = tokenizer(' '.join(str(row.text).split()), max_length=MAX_LEN,
                      truncation=True, padding='max_length', return_attention_mask=True)
        return {'ids':torch.tensor(t['input_ids']),
                'mask':torch.tensor(t['attention_mask']),
                'target':torch.tensor(int(row.label))}

def loaders():
    return {
        'train':DataLoader(WataDataset(train),batch_size=TRAIN_BS,shuffle=True),
        'train_eval':DataLoader(WataDataset(train),batch_size=EVAL_BS),
        'val':DataLoader(WataDataset(val),batch_size=EVAL_BS),
        'test':DataLoader(WataDataset(test),batch_size=EVAL_BS),
    }

class SoraniHybrid(nn.Module):
    def __init__(self):
        super().__init__()
        self.roberta = XLMRobertaModel.from_pretrained(MODEL)
        self.enc_dropout = nn.Dropout(0.05)
        self.lstm = nn.LSTM(self.roberta.config.hidden_size,256,bidirectional=True,batch_first=True)
        self.attention = nn.Linear(512,1)
        self.norm = nn.LayerNorm(512)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.28)
        self.head = nn.Linear(512,5)
    def forward(self,ids,mask):
        x = self.enc_dropout(self.roberta(input_ids=ids,attention_mask=mask).last_hidden_state)
        packed = nn.utils.rnn.pack_padded_sequence(x,mask.sum(1).cpu().long(),
                                                    batch_first=True,enforce_sorted=False)
        x,_ = self.lstm(packed)
        x,_ = nn.utils.rnn.pad_packed_sequence(x,batch_first=True,total_length=ids.size(1))
        scores = self.attention(x).squeeze(-1).masked_fill(~mask.bool(),torch.finfo(x.dtype).min)
        pooled = (x*torch.softmax(scores,dim=1).unsqueeze(-1)).sum(dim=1)
        return self.head(self.dropout(self.relu(self.norm(pooled))))

def seed_all(seed):
    os.environ['PYTHONHASHSEED']=str(seed)
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic=True
    torch.backends.cudnn.benchmark=False

def trainability(model, epoch):
    for p in model.roberta.parameters(): p.requires_grad=False
    if epoch>=2:
        for layer in model.roberta.encoder.layer[-8:]:
            for p in layer.parameters(): p.requires_grad=True

def optimizer_for(model):
    groups=[]
    encoder=list(model.roberta.named_parameters())
    head=[(n,p) for n,p in model.named_parameters() if not n.startswith('roberta.')]
    for params,lr in ((encoder,ENCODER_LR),(head,HEAD_LR)):
        for decay in (True,False):
            selected=[p for n,p in params if (not (n.endswith('bias') or 'norm' in n.lower()))==decay]
            groups.append({'params':selected,'lr':lr,'weight_decay':WEIGHT_DECAY if decay else 0.0})
    return torch.optim.AdamW(groups)

def scheduler_for(opt,batches):
    steps=EPOCHS*batches; warmup=max(1,int(WARMUP*steps))
    def scale(step):
        if step<warmup: return max(1e-5,(step+1)/warmup)
        progress=min(1.,(step-warmup)/max(1,steps-warmup))
        return .20+.80*.5*(1+math.cos(math.pi*progress))
    return torch.optim.lr_scheduler.LambdaLR(opt,scale)


In [ ]:
@torch.no_grad()
def evaluate(model,loader,loss_fn):
    model.eval()
    y_true=[]; y_pred=[]; probs=[]
    loss_total=mse_total=count=0
    for batch in loader:
        ids,mask,y=(batch[k].to(DEVICE) for k in ('ids','mask','target'))
        logits=model(ids,mask); p=logits.softmax(-1); n=len(y)
        loss_total+=loss_fn(logits,y).item()*n
        mse_total+=F.mse_loss(p,F.one_hot(y,5).float()).item()*n
        count+=n
        y_true.extend(y.cpu().tolist()); y_pred.extend(logits.argmax(-1).cpu().tolist())
        probs.extend(p.cpu().tolist())
    return {'accuracy':100*accuracy_score(y_true,y_pred),
            'macro_f1':f1_score(y_true,y_pred,labels=list(range(5)),average='macro',zero_division=0),
            'weighted_f1':f1_score(y_true,y_pred,labels=list(range(5)),average='weighted',zero_division=0),
            'loss':loss_total/count,'mse':mse_total/count,
            'y_true':np.asarray(y_true),'y_pred':np.asarray(y_pred),'probabilities':np.asarray(probs)}

def train_epoch(model,loader,loss_fn,opt,scheduler,epoch):
    model.train()
    if epoch==1: model.roberta.eval()
    for batch in loader:
        ids,mask,y=(batch[k].to(DEVICE) for k in ('ids','mask','target'))
        opt.zero_grad(set_to_none=True)
        loss=loss_fn(model(ids,mask),y)
        loss.backward()
        nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad],1.0)
        opt.step(); scheduler.step()


In [ ]:
all_results=[]
for seed in SEEDS:
    seed_all(seed)
    run_dir=OUT/f'seed_{seed}'
    run_dir.mkdir(exist_ok=True)
    dl=loaders()
    model=SoraniHybrid().to(DEVICE)
    opt=optimizer_for(model)
    schedule=scheduler_for(opt,len(dl['train']))
    criterion=nn.CrossEntropyLoss(weight=torch.tensor(weights,dtype=torch.float32,device=DEVICE),
                                  label_smoothing=SMOOTHING)
    best_f1,best_loss,best_epoch=-1.,float('inf'),None
    path=run_dir/'best_validation_checkpoint.pt'
    history=[]
    for epoch in range(1,EPOCHS+1):
        trainability(model,epoch)
        train_epoch(model,dl['train'],criterion,opt,schedule,epoch)
        tr=evaluate(model,dl['train_eval'],criterion)
        va=evaluate(model,dl['val'],criterion)
        history.append({'epoch':epoch,'train_accuracy':tr['accuracy'],'val_accuracy':va['accuracy'],
                        'train_loss':tr['loss'],'val_loss':va['loss'],'val_macro_f1':va['macro_f1']})
        if va['macro_f1']>best_f1+1e-12 or (abs(va['macro_f1']-best_f1)<=1e-12 and va['loss']<best_loss):
            best_f1,best_loss,best_epoch=va['macro_f1'],va['loss'],epoch
            torch.save(model.state_dict(),path)
        print(f'Seed {seed} epoch {epoch}: val macro-F1={va["macro_f1"]:.4f}')
    pd.DataFrame(history).to_csv(run_dir/'epoch_history.csv',index=False)
    del model,opt,schedule
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    model=SoraniHybrid().to(DEVICE)
    model.load_state_dict(torch.load(path,map_location=DEVICE,weights_only=True))
    result=evaluate(model,dl['test'],criterion)
    all_results.append({'seed':seed,'best_epoch':best_epoch,
                        **{k:result[k] for k in ('accuracy','macro_f1','weighted_f1','loss','mse')}})
    pd.DataFrame({'source_row':test['_source_row'],'true_label':result['y_true'],
                  'predicted_label':result['y_pred'],
                  **{f'prob_{i}':result['probabilities'][:,i] for i in range(5)}}).to_csv(
                      run_dir/'test_predictions.csv',index=False)
    (run_dir/'classification_report.txt').write_text(
        classification_report(result['y_true'],result['y_pred'],labels=list(range(5)),
                              target_names=LABELS,zero_division=0),encoding='utf-8')
    if seed==3407:
        history_plot=pd.DataFrame(history)
        result_plot=result
    del model
    if torch.cuda.is_available(): torch.cuda.empty_cache()

results=pd.DataFrame(all_results)
results.to_csv(OUT/'five_seed_results.csv',index=False)
summary={m:{'mean':float(results[m].mean()),'std':float(results[m].std(ddof=1))}
         for m in ('accuracy','macro_f1','weighted_f1')}
(OUT/'five_seed_summary.json').write_text(json.dumps(summary,indent=2),encoding='utf-8')
print(results.to_string(index=False)); print(summary)


In [ ]:
for metric,label in [('accuracy','Accuracy (%)'),('loss','Loss')]:
    fig,ax=plt.subplots(figsize=(8,4.5))
    ax.plot(history_plot['epoch'],history_plot[f'train_{metric}'],label='Training')
    ax.plot(history_plot['epoch'],history_plot[f'val_{metric}'],'--',label='Validation')
    ax.set(xlabel='Epoch',ylabel=label,xlim=(1,EPOCHS)); ax.set_ylim(bottom=0); ax.legend()
    fig.tight_layout(); fig.savefig(OUT/f'seed_3407_{metric}.png',dpi=180); plt.show()
ConfusionMatrixDisplay.from_predictions(result_plot['y_true'],result_plot['y_pred'],
    labels=np.arange(5),display_labels=LABELS,cmap='Blues',xticks_rotation=45)
plt.tight_layout(); plt.savefig(OUT/'seed_3407_confusion_matrix.png',dpi=180); plt.show()
truth=label_binarize(result_plot['y_true'],classes=np.arange(5))
fig,ax=plt.subplots(figsize=(8,5))
for i,label in enumerate(LABELS):
    fpr,tpr,_=roc_curve(truth[:,i],result_plot['probabilities'][:,i])
    ax.plot(fpr,tpr,label=f'{label} (AUC={auc(fpr,tpr):.3f})')
ax.plot([0,1],[0,1],'--')
ax.set(xlabel='False Positive Rate',ylabel='True Positive Rate',title='One-vs-Rest ROC')
ax.legend(); fig.tight_layout(); fig.savefig(OUT/'seed_3407_roc.png',dpi=180); plt.show()
